In [3]:
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from tqdm import tqdm

In [7]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

train_df_small = train_df.iloc[:10000]
val_df_small = val_df.iloc[:2000]
test_df_small = test_df.iloc[:2000]

In [8]:
def tokenizer(code):
    return code.split()

In [9]:
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

counter = Counter()

for _, row in train_df_small.iterrows():
    counter.update(tokenizer(row["buggy"]))
    counter.update(tokenizer(row["fixed"]))

vocab = {
    PAD: 0,
    SOS: 1,
    EOS: 2,
    UNK: 3
}

for token, _ in counter.items():
    vocab[token] = len(vocab)

words = {idx: word for word, idx in vocab.items()}

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 415


In [ ]:
def encode(tokens):
    ids = [vocab[SOS]]
    for token in tokens:
        ids.append(vocab.get(token, vocab[UNK]))
    ids.append(vocab[EOS])
    return torch.tensor(ids)

In [ ]:
class CodeFix(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        source = encode(tokenizer(row["buggy"]))
        target = encode(tokenizer(row["fixed"]))

        return source, target

In [ ]:
PAD_IDX = vocab[PAD]

def padder(batch):
    src_batch = [item[0] for item in batch]
    trg_batch = [item[1] for item in batch]

    src_batch = pad_sequence(src_batch,padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch,padding_value=PAD_IDX)

    return src_batch, trg_batch

In [ ]:
class Encoder(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embedding_dim,hidden_dim,num_layers)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)

        return hidden, cell

In [ ]:
class Decoder(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embedding_dim,hidden_dim,num_layers)
        self.fc = nn.Linear(hidden_dim,vocab_size)

    def forward(self,input_token,hidden,cell):
        input_token = input_token.unsqueeze(0)
        embedded = self.embedding(input_token)
        output, (hidden, cell) = self.lstm(embedded,(hidden, cell))
        prediction = self.fc(output.squeeze(0))

        return prediction, hidden, cell

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available()else "cpu")

class Seq2Seq(nn.Module):
    def __init__(self,encoder,decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self,src,trg,teacher_forcing_ratio=0.5):

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(trg_len,batch_size,vocab_size,device=src.device)
        hidden, cell = self.encoder(src)
        input_token = trg[0]
        for t in range(1, trg_len):
            prediction, hidden, cell = self.decoder(input_token,hidden,cell)
            outputs[t] = prediction
            best_guess = prediction.argmax(1)
            teacher_force = (random.random() < teacher_forcing_ratio)
            input_token = (trg[t] if teacher_force else best_guess)
            
        return outputs

In [ ]:
VOCAB_SIZE = len(vocab)
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
# train_dataset = CodeFix(train_df)
# val_dataset = CodeFix(val_df)
# test_dataset = CodeFix(test_df)
train_dataset = CodeFix(train_df_small)
val_dataset = CodeFix(val_df_small)
test_dataset = CodeFix(test_df_small)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,collate_fn=padder)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,collate_fn=padder)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,collate_fn=padder)

In [ ]:
def train(model,loader,optimizer,criterion):
    model.train()
    epoch_loss = 0
    for src, trg in tqdm(loader):
        src = src.to(device)
        trg = trg.to(device)
        optimizer.zero_grad()
        output = model(src,trg)
        output = output[1:].reshape(-1,output.shape[-1])
        trg = trg[1:].reshape(-1)
        loss = criterion(output,trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [ ]:
def validate(model,loader,criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src = src.to(device)
            trg = trg.to(device)
            output = model(src,trg,teacher_forcing_ratio=0)
            output = output[1:].reshape(-1,output.shape[-1])
            trg = trg[1:].reshape(-1)
            loss = criterion(output,trg)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [ ]:
def repair(code,model,max_len=100):
    model.eval()
    tokens = tokenizer(code)
    src = encode(tokens).unsqueeze(1).to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(src)
        input_token = torch.tensor([vocab[SOS]],dtype=torch.long,device=device)
        generated = []
        for _ in range(max_len):
            prediction, hidden, cell = model.decoder(input_token,hidden,cell)
            best_guess = prediction.argmax(1)
            token_id = best_guess.item()
            if token_id == vocab[EOS]:
                break
            generated.append(words[token_id])
            input_token = best_guess

    return " ".join(generated)

In [ ]:
def run(model,train_loader,val_loader,epochs=5,lr=0.001,model_name="Model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = optim.Adam(model.parameters(),lr=lr)
    train_losses = []
    val_losses = []
    for epoch in range(epochs):
        train_loss = train(model,train_loader,optimizer,criterion)
        val_loss = validate(model,val_loader,criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"{model_name} Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return model, train_losses, val_losses

In [ ]:
def evaluate(model,dataframe,samples=1000):
    model.eval()
    correct_tokens = 0
    total_tokens = 0
    for i in range(min(samples, len(dataframe))):
        row = dataframe.iloc[i]
        pred = repair(row["buggy"],model).split()
        actual = row["fixed"].split()
        m = min(len(pred),len(actual))
        for j in range(m):
            if pred[j] == actual[j]:
                correct_tokens += 1
        total_tokens += max(len(pred),len(actual))
    return (correct_tokens/total_tokens)

In [ ]:
encoder = Encoder(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS)
decoder = Decoder(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS)
baseline_model = Seq2Seq(encoder,decoder)
baseline_model, train_hist, val_hist = run(baseline_model,train_loader,val_loader,epochs=5,model_name="Baseline")
baseline_acc = evaluate(baseline_model,val_df,1000)
print("Baseline Token Accuracy:",baseline_acc * 100)

torch.save(baseline_model.state_dict(),"baseline_lstm.pth")

In [ ]:
class BiEncoder(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embedding_dim,hidden_dim,num_layers,bidirectional=True)
    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = hidden[0:hidden.size(0):2] + hidden[1:hidden.size(0):2]
        cell = cell[0:cell.size(0):2] + cell[1:cell.size(0):2]
        return hidden, cell

In [ ]:
encoder = BiEncoder(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS)
decoder = Decoder(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS)
bilstm = Seq2Seq(encoder,decoder)
bilstm, train_hist, val_hist = run(bilstm,train_loader,val_loader,epochs=5,model_name="BiLSTM")
bilstm_acc = evaluate(bilstm,val_df,1000)
print("BiLSTM Token Accuracy:",bilstm_acc * 100)

torch.save(bilstm.state_dict(),"bilstm.pth")

In [ ]:
class BiEncoderDropout(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_layers=1,dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=PAD_IDX)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embedding_dim,hidden_dim,num_layers,bidirectional=True)

    def forward(self, src):
        embedded = self.embedding(src)
        embedded = self.dropout(embedded)
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = (hidden[0:hidden.size(0):2]+hidden[1:hidden.size(0):2])
        cell = (cell[0:cell.size(0):2]+cell[1:cell.size(0):2])

        return hidden, cell

In [ ]:
encoder = BiEncoderDropout(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS,dropout=0.3)
decoder = Decoder(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS)
dropout_model = Seq2Seq(encoder,decoder)
dropout_model, train_hist, val_hist = run(dropout_model,train_loader,val_loader,epochs=5,model_name="BiLSTM+Dropout")
dropout_acc = evaluate(dropout_model,val_df,1000)
print("BiLSTM + Dropout Token Accuracy:",dropout_acc * 100)

torch.save(dropout_model.state_dict(),"bilstm_dropout.pth")

In [1]:
class Attention(nn.Module):
        def __init__(self, hidden_dim):
                super().__init__()
                self.attn = nn.Linear(hidden_dim * 3,hidden_dim)
                self.v = nn.Linear(hidden_dim,1,bias=False)
                                   
        def forward(self,hidden,encoder_outputs):
                src_len = encoder_outputs.shape[0]
                hidden = hidden.repeat(src_len,1,1)
                energy = torch.tanh(self.attn(torch.cat((hidden,encoder_outputs),dim=2)))
                attention = self.v(energy).squeeze(2)
                return torch.softmax(attention,dim=0)

NameError: name 'nn' is not defined

In [ ]:
class AttentionBiEncoder(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embedding_dim,hidden_dim,num_layers,bidirectional=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = (hidden[0:hidden.size(0):2]+hidden[1:hidden.size(0):2])
        cell = (cell[0:cell.size(0):2]+cell[1:cell.size(0):2])

        return outputs, hidden, cell

In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=PAD_IDX)
        self.attention = attention
        self.lstm = nn.LSTM(embedding_dim + hidden_dim * 2,hidden_dim)
        self.fc = nn.Linear(hidden_dim,vocab_size)
    
    def forward(self,input_token,hidden,cell,encoder_outputs):
        input_token = input_token.unsqueeze(0)
        embedded = self.embedding(input_token)
        attn_weights = self.attention(hidden[-1].unsqueeze(0),encoder_outputs)
        context = torch.sum(attn_weights.unsqueeze(2)*encoder_outputs,dim=0)
        context = context.unsqueeze(0)
        lstm_input = torch.cat((embedded,context),dim=2)
        output, (hidden, cell) = self.lstm(lstm_input,(hidden, cell))
        prediction = self.fc(output.squeeze(0))

        return prediction, hidden, cell

In [ ]:
class AttentionSeq2Seq(nn.Module):

    def __init__(self,encoder,decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self,src,trg,teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(trg_len,batch_size,vocab_size,device=src.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input_token = trg[0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token,hidden,cell,encoder_outputs)
            outputs[t] = output
            best_guess = output.argmax(1)
            teacher_force = (random.random()<teacher_forcing_ratio)
            input_token = ( trg[t] if teacher_force else best_guess )

        return outputs

In [ ]:
def repair_attention(code,model,max_len=100):
    model.eval()
    tokens = tokenizer(code)
    src = encode(tokens).unsqueeze(1).to(device)
    with torch.no_grad():
        encoder_outputs, hidden, cell = (model.encoder(src))
        input_token = torch.tensor([vocab[SOS]],dtype=torch.long,device=device)
        generated = []
        for _ in range(max_len):
            prediction, hidden, cell = (model.decoder(input_token,hidden,cell,encoder_outputs))
            best_guess = prediction.argmax(1)
            token_id = best_guess.item()
            if token_id == vocab[EOS]:
                break
            generated.append(words[token_id])
            input_token = best_guess

    return " ".join(generated)

In [ ]:
def evaluate_attention(model,dataframe,samples=1000):
    model.eval()
    correct_tokens = 0
    total_tokens = 0
    for i in range(min(samples, len(dataframe))):
        row = dataframe.iloc[i]
        pred = repair_attention(row["buggy"],model).split()
        actual = row["fixed"].split()
        m = min(len(pred),len(actual))
        for j in range(m):
            if pred[j] == actual[j]:
                correct_tokens += 1
        total_tokens += max(len(pred),len(actual))

    return (correct_tokens / total_tokens)

In [ ]:
attention = Attention(HIDDEN_DIM)
encoder = AttentionBiEncoder(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,NUM_LAYERS)
decoder = AttentionDecoder(VOCAB_SIZE,EMBED_DIM,HIDDEN_DIM,attention)
attention_model = AttentionSeq2Seq(encoder,decoder)
attention_model, train_hist, val_hist = run(attention_model,train_loader,val_loader,epochs=5,model_name="BiLSTM + Attention")
attention_acc = evaluate_attention(attention_model,val_df,1000)
print("BiLSTM + Attention Token Accuracy:",attention_acc * 100)

torch.save(attention_model.state_dict(),"bilstm_attention.pth")

In [11]:
torch.save(vocab, "base_vocab.pt")
torch.save(words, "base_words.pt")

### Experimental Results (10k Training Samples, 2k Validation Samples, 5 Epochs)

| Model              | Train Loss | Validation Loss | Token Accuracy (%) |
| ------------------ | ---------- | --------------- | ------------------ |
| LSTM               | 1.6580     | 2.8098          | 31.88              |
| BiLSTM             | 1.4797     | 2.5878          | 35.21              |
| BiLSTM + Dropout   | 1.5443     | 2.6170          | 34.71              |
| BiLSTM + Attention | 0.9509     | 2.1836          | 52.10              |

### Observations

1. **BiLSTM vs LSTM**

   * Using a bidirectional encoder improved token accuracy from 31.88% to 35.21%.
   * Validation loss also decreased from 2.8098 to 2.5878.
   * This suggests that incorporating both past and future context helps the model capture code dependencies more effectively.

2. **Effect of Dropout**

   * Adding dropout slightly reduced token accuracy from 35.21% to 34.71%.
   * Validation loss showed negligible change.
   * This indicates that the model was not significantly overfitting and additional regularization was not beneficial at this scale.

3. **Effect of Attention**

   * Attention produced the largest improvement among all experiments.
   * Token accuracy increased from 35.21% to 52.10%.
   * Validation loss dropped substantially from 2.5878 to 2.1836.
   * The attention mechanism allowed the decoder to directly access relevant encoder outputs instead of relying solely on the final hidden state, which is particularly useful for code repair tasks where most of the source sequence must be copied while only a few tokens are modified.

### Conclusion

The strongest recurrent architecture validated was the BiLSTM with Attention model. The experiments show that improving access to source-code information is more important than regularization for this task. While BiLSTM provided modest gains, the attention mechanism resulted in a significant improvement in code-repair performance.
